---
## Короткі теоретичні відомості

**Орбіталі.** *Атомна орбіталь* (АО) — базисна функція, з якої будується молекулярна/атомна орбіталь;
у PySCF це гаусові функції базисного набору. *Молекулярна (атомна) орбіталь* (МО) $\varphi_i(\mathbf{r})$ —
одноелектронна просторова хвильова функція, яка є розв'язком рівнянь Хартрі–Фока і подається як лінійна
комбінація АО: $\varphi_i(\mathbf{r}) = \sum_\mu C_{\mu i}\,\chi_\mu(\mathbf{r})$.

**Спін-орбіталі.** Електрон описується не лише просторовою координатою, а й спіном. *Спін-орбіталь*
$\chi_i(\mathbf{x})$, $\mathbf{x}=(\mathbf{r},\omega)$, — це добуток просторової орбіталі та спінової функції:
$$\chi_i(\mathbf{x}) = \varphi_i(\mathbf{r})\,\alpha(\omega) \quad\text{або}\quad \chi_i(\mathbf{x}) = \varphi_i(\mathbf{r})\,\beta(\omega),$$
де $\alpha$ відповідає спіну "вгору" ($m_s=+\tfrac12$), $\beta$ — спіну "вниз" ($m_s=-\tfrac12$). Одна й та сама
просторова орбіталь може бути "заселена" двома електронами лише з протилежними спінами (принцип Паулі).

**Детермінант Слейтера.** Багатоелектронна хвильова функція повинна бути антисиметричною відносно перестановки
будь-яких двох електронів (принцип Паулі). Для системи з $N$ електронів, які займають спін-орбіталі
$\chi_1,\ldots,\chi_N$, таку антисиметричну функцію будують у вигляді визначника:
$$\Psi(\mathbf{x}_1,\ldots,\mathbf{x}_N) = \frac{1}{\sqrt{N!}}
\begin{vmatrix}
\chi_1(\mathbf{x}_1) & \chi_2(\mathbf{x}_1) & \cdots & \chi_N(\mathbf{x}_1)\\
\chi_1(\mathbf{x}_2) & \chi_2(\mathbf{x}_2) & \cdots & \chi_N(\mathbf{x}_2)\\
\vdots & \vdots & \ddots & \vdots\\
\chi_1(\mathbf{x}_N) & \chi_2(\mathbf{x}_N) & \cdots & \chi_N(\mathbf{x}_N)
\end{vmatrix}.$$
Перестановка двох рядків (тобто двох електронів) змінює знак визначника — це і забезпечує антисиметрію.
Якщо два електрони мають однакову спін-орбіталь (два однакові стовпці), визначник дорівнює нулю — це і є
принцип Паулі в явному вигляді.

**Типи методів Хартрі–Фока.**
- **RHF** (Restricted HF) — для систем із замкненими оболонками ($N_\alpha = N_\beta$): кожна просторова
  орбіталь двічі заселена електронами з протилежними спінами, тобто $\varphi_i^\alpha \equiv \varphi_i^\beta$.
- **UHF** (Unrestricted HF) — для відкритих оболонок ($N_\alpha \neq N_\beta$): $\alpha$- та $\beta$-електрони
  мають окремі, незалежні просторові орбіталі. Простіший у реалізації, але може порушувати спінову симетрію
  (*spin contamination*: $\langle S^2\rangle$ відхиляється від теоретичного $S(S+1)$).
- **ROHF** (Restricted Open-shell HF) — компроміс: замкнені підоболонки мають спільні просторові орбіталі
  для $\alpha$ і $\beta$, а відкриті (неспарені) орбіталі — окремі. Зберігає точне значення $\langle S^2\rangle$,
  але рівняння Хартрі-Фока складніші для розв'язання.


# Атоми Li та Ar методом Хартрі–Фока (PySCF)

**Мета роботи:**

**Частина 1 (Li, Z=3):**
1. Визначення терму основного стану за правилами Гунда.
2. Запис вигляду детермінанта Слейтера для конфігурації $1s^2 2s^1$.
3. Розрахунок методами UHF та ROHF, порівняння $\langle S^2\rangle$.
4. Дослідження збіжності енергії за базисним набором.
5. Аналіз орбітального спектра ($\alpha$- та $\beta$-орбіталі) і матриці густини.
6. Перевірка теореми віріала.
7. Оцінка кореляційної енергії (порівняння з експериментом).
8. Побудова радіальної функції розподілу густини, ідентифікація оболонок $1s$, $2s$.

**Частина 2 (Ar, Z=18) — тільки:**
1. RHF-розрахунок.
2. Побудова сферично усередненої радіальної функції розподілу $4\pi r^2\rho(r)$.
3. Визначення кількості максимумів (оболонок).

> Усі клітинки з позначкою `# TODO` потрібно заповнити самостійно. Готового коду з відповідями тут немає —
> орієнтуйтесь на приклади з попередньої роботи (атом He) та на формули з лекцій.
>
> `pip install pyscf` — якщо пакет ще не встановлено.


In [ ]:
!pip install pyscf

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

from pyscf import gto, scf, ao2mo

np.set_printoptions(precision=6, suppress=True)


---
# Частина 1. Атом Літію (Li, Z = 3)

Електронна конфігурація: $1s^2\,2s^1$.


## Завдання 1.1. Терм основного стану

Терм ${}^{2S+1}L_J$ — це компактний запис квантового стану атома через **сумарні** квантові числа
$S$ (повний спін), $L$ (повний орбітальний момент) і $J$ (повний момент), а не просто перелік
заповнених орбіталей. Він показує, у якому саме стані з-поміж кількох можливих реально перебуває
атом, коли електронна конфігурація допускає не один розподіл по орбіталях.

Знати терм важливо для подальшої роботи:
- він одразу підказує, який метод SCF використовувати: $S=0$ → RHF, $S\neq0$ → UHF/ROHF;
- теоретичне значення $S(S+1)$ дає орієнтир, з яким можна звірити $\langle S^2\rangle$ з розрахунку
  (перевірка на spin contamination);
- саме терми (а не окремі орбіталі) відповідають рівням, які спостерігаються в атомних спектрах.

За правилами Гунда визначте терм основного стану ${}^{2S+1}L_J$ атома Li.

Запишіть у клітинці нижче:
- квантові числа $S$ і $L$;
- символ терму;
- одним реченням — фізичне пояснення.

**Ваша відповідь:**

- $S = ...$
- $L = ...$
- Терм: ...
- Пояснення: ...


## Завдання 1.2. Детермінант Слейтера для Li ($1s^2 2s^1$)

У роботі з атомом He ви будували двоелектронний детермінант Слейтера для конфігурації $1s^2$:
$$\Psi(\mathbf{x}_1,\mathbf{x}_2) = \frac{1}{\sqrt{2}}
\begin{vmatrix}
\chi_1(\mathbf{x}_1) & \chi_2(\mathbf{x}_1) \\
\chi_1(\mathbf{x}_2) & \chi_2(\mathbf{x}_2)
\end{vmatrix}, \qquad
\chi_1(\mathbf{x}) = \varphi_{1s}(\mathbf{r})\alpha(\omega), \quad
\chi_2(\mathbf{x}) = \varphi_{1s}(\mathbf{r})\beta(\omega).$$

**Завдання:** атом Li має **три** електрони: два на $1s$ (з протилежними спінами) і один — неспарений — на $2s$.

1. Випишіть явний вигляд усіх трьох спін-орбіталей $\chi_1(\mathbf{x})$, $\chi_2(\mathbf{x})$, $\chi_3(\mathbf{x})$
   через просторові орбіталі $\varphi_{1s}(\mathbf{r})$, $\varphi_{2s}(\mathbf{r})$ та спінові функції $\alpha(\omega)$, $\beta(\omega)$
   (для неспареного електрона на $2s$ візьміть спін $\alpha$ — саме такий вибір відповідає $M_S=+\tfrac12$).
2. Запишіть повний тритрьохелектронний детермінант Слейтера $\Psi(\mathbf{x}_1,\mathbf{x}_2,\mathbf{x}_3)$
   у вигляді визначника $3\times3$ з нормувальним множником (яким він має бути для 3 електронів?).
3. За бажанням — розпишіть цей визначник у вигляді суми $3! = 6$ доданків зі знаками (аналогічно до того,
   як двоелектронний визначник розписується у два доданки).

**Ваша відповідь:**

$\chi_1(\mathbf{x}) = ...$

$\chi_2(\mathbf{x}) = ...$

$\chi_3(\mathbf{x}) = ...$

$$\Psi(\mathbf{x}_1,\mathbf{x}_2,\mathbf{x}_3) = \dfrac{1}{...}
\begin{vmatrix}
... & ... & ... \\
... & ... & ... \\
... & ... & ...
\end{vmatrix}$$

(за бажанням) розгорнутий вигляд через 6 доданків:

$\Psi = ...$


## Завдання 1.3. UHF-розрахунок Li

**Мікропитання:** скільки електронів Li мають спін $\alpha$, а скільки $\beta$? Який параметр `spin`
потрібно передати в `gto.M(...)`? Чому в цьому випадку недостатньо `scf.RHF`?

Побудуйте `mol` для атома Li (базис `cc-pVDZ`) з відповідним значенням `spin`, виконайте розрахунок
методом **UHF**, виведіть повну енергію та $\langle S^2\rangle$ (через `mf.spin_square()`).

In [ ]:
basis = 'cc-pvdz'

# TODO 1: mol_li = gto.M(atom='Li 0 0 0', basis=basis, spin=???, charge=0, verbose=0)

# TODO 2: mf_uhf = scf.UHF(mol_li); e_uhf = mf_uhf.kernel()

# TODO 3: s2_uhf, mult_uhf = mf_uhf.spin_square()

# TODO 4: виведіть E(UHF), <S^2>, мультиплетність


## Завдання 1.4. Метод ROHF — порівняння з UHF

**Мікропитання:** як обрати інший метод SCF, лишивши той самий об'єкт `mol`? Чим ROHF відрізняється
від UHF у побудові орбіталей для замкнених/відкритих підоболонок?

Проведіть розрахунок Li методом **ROHF** у тому самому базисі, виведіть $E$, $\langle S^2\rangle$,
мультиплетність.

In [ ]:
# TODO 1: mf_rohf = scf.ROHF(mol_li); e_rohf = mf_rohf.kernel()

# TODO 2: s2_rohf, mult_rohf = mf_rohf.spin_square()

# TODO 3: виведіть результати у такому ж форматі, як для UHF


**Порівняйте отримані результати й дайте відповідь:**

- Наскільки відрізняються $E_{UHF}$ і $E_{ROHF}$?
- Наскільки $\langle S^2\rangle_{UHF}$ відхиляється від теоретичного значення $S(S+1)$ (spin contamination),
  і чи є таке відхилення у ROHF?
- Який метод дає нижчу енергію і чому?

**Ваша відповідь:** ...


## Завдання 1.5. Збіжність енергії за базисним набором

Дослідіть, як енергія UHF атома Li залежить від розміру базисного набору.

Заповніть список `basis_list` кількома базисами зростаючого розміру і побудуйте графік $E(\mathrm{n_{ao}})$,
як у роботі з He. Для порівняння: експериментальна (нерелятивістська) енергія основного стану Li
$\approx -7.478$ Hartree.

In [ ]:
basis_list = []  # TODO: додайте базиси, наприклад 'sto-3g', '6-31g', '6-31g**', 'cc-pvdz', 'cc-pvtz', ...

results_li = {}
for basis in basis_list:
    # TODO 1: побудуйте mol для Li з поточним basis (той самий spin, що й у Завданні 1.3)

    # TODO 2: виконайте UHF розрахунок

    # TODO 3: збережіть результат у results_li[basis] = dict(mol=mol, mf=mf, e_tot=e_tot)

    # TODO 4: виведіть рядок з назвою базису, nao та енергією
    pass

# TODO 5: побудуйте графік E(nao); додайте горизонтальну лінію на рівні -7.478 Hartree для порівняння


## Завдання 1.6. Орбітальний спектр: $\alpha$- та $\beta$-орбіталі

**Мікропитання:** якої структури дані повертають `mf.mo_energy` та `mf.mo_occ` для UHF-об'єкта?
(Перевірте самі — виведіть `type(mf_uhf.mo_energy)` та `len(mf_uhf.mo_energy)`.)

Виведіть таблицю орбітальних енергій окремо для $\alpha$ та $\beta$ спінів (номер MO, енергія, occ,
зайнята/віртуальна), як робили для He.

In [ ]:
# TODO 1: перевірте структуру mf_uhf.mo_energy і mf_uhf.mo_occ (print(type(...)), print(len(...)))

# TODO 2: виведіть таблицю alpha-орбіталей

# TODO 3: виведіть таблицю beta-орбіталей

# TODO 4: порівняйте енергію 1s-орбіталі для alpha і beta. Чому вони не однакові,
#         хоча обидва електрони формально належать до тієї самої 1s-підоболонки?


## Завдання 1.7. Матриця густини та перевірка сліду

Матриця густини $D_{\mu\nu}$ (в АО-базисі) містить у собі всю інформацію про розподіл електронів у
даному розрахунку: саме через неї обчислюється електронна густина в будь-якій точці простору,
$\rho(\mathbf{r}) = \sum_{\mu\nu}\phi_\mu(\mathbf{r})D_{\mu\nu}\phi_\nu(\mathbf{r})$. Її **слід** $\mathrm{Tr}(D)$ — це
інтеграл густини по всьому простору, тобто повна кількість електронів, яку "бачить" ця матриця. Тому
перевірка сліду — простий, але важливий тест: якщо $\mathrm{Tr}(D)$ не збігається з очікуваною
кількістю електронів, це означає помилку в побудові густини (неправильний базис, неправильно
зайняті орбіталі тощо).

**Завдання:**
1. Отримайте `dm_a, dm_b = mf_uhf.make_rdm1()`.
2. Обчисліть сліди $\mathrm{Tr}(dm_\alpha)$, $\mathrm{Tr}(dm_\beta)$, $\mathrm{Tr}(dm_\alpha+dm_\beta)$.
3. Порівняйте з `mol_li.nelec` (кортеж $(N_\alpha, N_\beta)$) — чи виконується очікувана рівність?

In [ ]:
# TODO 1: dm_a, dm_b = mf_uhf.make_rdm1()

# TODO 2: обчисліть сліди

# TODO 3: виведіть результат і порівняйте з mol_li.nelec


## Завдання 1.8. Теорема віріала для Li

$$-\frac{\langle V\rangle}{\langle T\rangle} = 2.$$

Кроки:
1. $dm_{total} = dm_\alpha + dm_\beta$.
2. $T = \mathrm{Tr}\big(dm_{total}\cdot T_{int}\big)$, де `T_int = mol.intor('int1e_kin')`.
3. $E_{elec} = E_{tot} - E_{nuc}$ (для одного атома $E_{nuc}=0$).
4. $V = E_{elec} - T$.
5. Перевірте $-V/T$.

In [ ]:
# TODO 1: dm_total = dm_a + dm_b

# TODO 2: T_int = mol_li.intor('int1e_kin'); T = ...

# TODO 3: E_elec = mf_uhf.e_tot - mol_li.energy_nuc()

# TODO 4: V = E_elec - T

# TODO 5: виведіть T, V, T+V, E_elec, -V/T (порівняйте з очікуваним значенням ~2)


## Завдання 1.9. Оцінка кореляційної енергії

Порівняйте отриману енергію $E_{UHF}$(Li, cc-pVDZ) з експериментальною $\approx -7.478$ Hartree.
Обчисліть різницю в Hartree та в еВ ($1\,\mathrm{Ha} = 27.2114\ \mathrm{eV}$).

In [ ]:
E_exp_li = -7.478  # Hartree, довідкове значення

# TODO 1: E_corr = E_exp_li - e_uhf

# TODO 2: переведіть E_corr у еВ

# TODO 3: print(...)


## Завдання 1.10. Радіальна функція розподілу для Li

Обидві зайняті просторові орбіталі Li ($1s$, $2s$) — $s$-типу, тобто сферично симетричні,
тому, як і для He, достатньо обчислити їх уздовж однієї осі.

**Кроки:**
1. Побудуйте сітку `r_points` уздовж осі $x$ від 0 до ~6 Bohr.
2. Обчисліть АО на сітці, а потім МО окремо для $\alpha$ і $\beta$ (`mf_uhf.mo_coeff[0]`, `mf_uhf.mo_coeff[1]`).
3. Для кожної зайнятої $\alpha$- і $\beta$-орбіталі побудуйте $4\pi r^2\varphi_i(r)^2$.
4. Визначте, скільки максимумів видно на графіку, і зіставте їх з $1s$ та $2s$.

In [ ]:
r = np.linspace(0.001, 6.0, 400)
coords = np.zeros((len(r), 3))
coords[:, 0] = r

ao_vals = mol_li.eval_gto('GTOval', coords)

# TODO 1: mo_vals_a = ao_vals @ mf_uhf.mo_coeff[0]
# TODO 2: mo_vals_b = ao_vals @ mf_uhf.mo_coeff[1]

# TODO 3: знайдіть індекси зайнятих alpha- і beta-орбіталей (mf_uhf.mo_occ)

# TODO 4: для кожної зайнятої alpha-орбіталі побудуйте графік 4*pi*r^2*phi^2

# TODO 5: те саме для зайнятої beta-орбіталі

# TODO 6: знайдіть r_max для кожної кривої (np.argmax) і випишіть висновок:
#         скільки максимумів (оболонок) видно й до яких орбіталей вони належать?


---
# Частина 2. Атом Аргону (Ar, Z = 18)

Електронна конфігурація: $1s^2\,2s^2\,2p^6\,3s^2\,3p^6$ — замкнена оболонка.

**Для Ar виконуємо тільки:**
1. RHF-розрахунок.
2. Побудову сферично усередненої радіальної функції розподілу $4\pi r^2\rho(r)$.
3. Визначення кількості максимумів (оболонок).

Розрахунок термів, порівняння UHF/ROHF, збіжність базису тощо для Ar не потрібні.

In [ ]:
basis_ar = 'cc-pvtz'

# TODO 1: mol_ar = gto.M(atom='Ar 0 0 0', basis=basis_ar, spin=???, charge=0, verbose=0)

# TODO 2: mf_ar = scf.RHF(mol_ar); e_ar = mf_ar.kernel()

# TODO 3: виведіть E(RHF) та mol_ar.nelectron


## Завдання 2.1. Радіальна функція розподілу $\rho(r)$

Так само, як ви робили для He та Li, обчисліть електронну густину вздовж однієї осі (наприклад $x$)
через матрицю густини: $\rho(\mathbf r) = \sum_{i,j}\phi_i(\mathbf r)\,D_{ij}\,\phi_j(\mathbf r)$,
де $D$ — матриця густини (`mf_ar.make_rdm1()`), а $\phi_i$ — значення АО в точці.

**Кроки:**
1. Побудуйте сітку `r_points` уздовж осі $x$ від 0 до ~8 Bohr.
2. Для кожного $r$ обчисліть АО в точці $(r,0,0)$ (`mol_ar.eval_gto(\'GTOval_sph\', coords)`) і густину $\rho(r)$ через `dm`.
3. Побудуйте радіальну функцію розподілу $4\pi r^2\rho(r)$.
4. Перевірте нормування: $\int 4\pi r^2\rho(r)\,dr$ має бути близько до кількості електронів Ar (18).

In [ ]:
r_ar = np.linspace(0.01, 8.0, 300)

dm_ar = mf_ar.make_rdm1()

rho_ar = []
for r in r_ar:
    coords = np.array([[r, 0.0, 0.0]])

    # TODO 1: ao_vals = mol_ar.eval_gto('GTOval_sph', coords)

    # TODO 2: обчисліть rho_point = np.einsum('pi,ij,pj->p', ao_vals, dm_ar, ao_vals)[0]

    # TODO 3: rho_ar.append(rho_point)
    pass

rho_ar = np.array(rho_ar)

# TODO 4: rdf_ar = 4 * np.pi * r_ar**2 * rho_ar

# TODO 5: перевірте нормування: np.trapz(rdf_ar, r_ar) має бути близько до mol_ar.nelectron


In [ ]:
# TODO: побудуйте графіки rho(r) (лог. масштаб по y) і радіальної функції 4*pi*r^2*rho(r)


## Завдання 2.2. Визначення кількості оболонок

Скористайтесь `scipy.signal.find_peaks` (уже імпортовано на початку роботи).

**Завдання:**
1. Знайдіть індекси локальних максимумів `rdf_ar`.
2. Виведіть кількість піків та значення `r`, при яких вони спостерігаються.
3. Порівняйте кількість знайдених піків із кількістю оболонок $K$ ($n=1$), $L$ ($n=2$), $M$ ($n=3$) і
   поясніть результат (чи всі підоболонки одного $n$ дають окремий пік, чи можуть зливатись).

In [ ]:
# TODO 1: peaks, _ = find_peaks(rdf_ar)

# TODO 2: виведіть кількість піків та r_ar[peaks]

# TODO 3: позначте максимуми на графіку rdf_ar


**Висновок (заповніть):**

- Скільки максимумів знайдено на кривій $4\pi r^2\rho(r)$ для Ar?
- Яким оболонкам ($K,L,M$) вони відповідають?
- Чи всі підоболонки одного $n$ дають окремий пік, чи деякі зливаються? Чому?

**Ваша відповідь:** ...
